In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torch.optim as optim

from torchvision import datasets, transforms

from sklearn.metrics import accuracy_score, classification_report


# Load and Preprocess

In [2]:
transform = transforms.Compose(
    [
        transforms.Resize((160, 160)),
        transforms.ToTensor(), # convert to tensore and min-max scale (0, 1)
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)), # normalize (-1, 1)
    ]
)

In [3]:
# Load dataset
dataset = datasets.ImageFolder(
    root="datasets/PetImages/",
    transform=transform
)

In [4]:
# Split into train and test
train_dataset, val_dataset, test_dataset = random_split(
    dataset=dataset,
    lengths=[0.7, 0.15, 0.15]
)


In [5]:
# Convert to DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=True
)

# CNN

## Building the model

In [6]:
# Define CNN
class CatDogCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # Image size: 540 x 540 x 3
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1), # 160 x 160 x 8
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 80 x 80 x 8

            nn.Conv2d(8, 16, kernel_size=3, padding=1), # 80 x 80 x 16
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 40 x 40 x 16

            nn.Conv2d(16, 32, kernel_size=3, padding=1), # 40 x 40 x 32
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 20 x 20 x 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 20 x 20 x 64
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 10 x 10 x 64

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 10 x 10 x 128
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 5 x 5 x 128

            nn.Conv2d(128, 256, kernel_size=3, padding=1), # 5 x 5 x 256
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 2 x 2 x 256
        )

        self.fc_layers = nn.Sequential(
            # receives flattened layer
            nn.Linear(2 * 2 * 256, 256),
            nn.ReLU(),

            nn.Linear(256, 1) # convert to final label either cat or dog
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattein to 1d
        x = self.fc_layers(x)
        return x

In [7]:
# Instantiate CNN
cnn_model = CatDogCNN()

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(cnn_model.parameters())


## Training

In [ ]:
epochs = 10
min_val_loss = float("inf")

for epoch in range(0, epochs):
    # Training
    cnn_model.train()
    train_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad() # remove previous gradients
        outputs = cnn_model.forward(images) # predict output
        outputs = outputs.squeeze() 
        labels = labels.float()
        loss = criterion(outputs, labels) # calculate loss
        loss.backward() # start backward propagation - cal gradients
        optimizer.step() # adjust parameters

        train_loss += loss.item()

    # Validation
    cnn_model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = cnn_model.forward(images)
            outputs = outputs.squeeze()
            labels = labels.float() 
            loss = criterion(outputs, labels)

            val_loss += loss.item()

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    # Choosing best model
    if val_loss < min_val_loss:
        min_val_loss = val_loss
        torch.save(cnn_model.state_dict(), "catdog_cnn.pt")

    print(f"Epoch: {epoch} Train => {train_loss} Val => {val_loss}")

c:\Users\User\.conda\envs\torch_env\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch: 0 Train => 187.67133992910385 Val => 39.405728816986084
Epoch: 1 Train => 173.58388009667397 Val => 33.882662415504456
Epoch: 2 Train => 141.19018244743347 Val => 30.48263618350029
Epoch: 3 Train => 118.43157167732716 Val => 27.896041467785835
Epoch: 4 Train => 101.6222950220108 Val => 22.510137885808945
Epoch: 5 Train => 87.9669037759304 Val => 21.864690706133842
Epoch: 6 Train => 77.61576802283525 Val => 20.107116743922234
Epoch: 7 Train => 67.95671223849058 Val => 19.822115197777748
Epoch: 8 Train => 56.35584680736065 Val => 21.982465773820877
Epoch: 9 Train => 48.808851052075624 Val => 21.118803665041924


## Evaluation

In [11]:
# Loading best model
cnn_model.load_state_dict(torch.load("catdog_cnn.pt"))

C:\Users\User\AppData\Local\Temp\ipykernel_24404\987957952.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load("catdog_cnn.pt"))


<All keys matched successfully>

In [ ]:
# Evaluation
cnn_model.eval()
y_actual = []
y_predicted = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = cnn_model.forward(images) # predict and get logits
        outputs = torch.sigmoid(outputs) # conver to probabilty range (0, 1)
        outputs = (outputs >= 0.5) # get labels

        y_actual.extend(labels.cpu().numpy())
        y_predicted.extend(outputs.cpu().numpy())

        

In [35]:
print(f"Accuracy: {accuracy_score(y_actual, y_predicted)}")
print(classification_report(y_actual, y_predicted))

Accuracy: 0.855428114163777
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      1848
           1       0.86      0.86      0.86      1901

    accuracy                           0.86      3749
   macro avg       0.86      0.86      0.86      3749
weighted avg       0.86      0.86      0.86      3749

